# Notebook 01 - Problem framing (Steps 1-2)

This notebook loads **no data**. It records the decisions we make *before* touching the data, so we cannot tune them afterwards to make the results look good. Every number about the data appears later, printed by code in notebooks 02-04.

## Step 1 - Understand the problem

### 1.1 The problem in plain words
Lending Club was an online lender. People applied for a personal loan; the lender decided whether to fund it and at what interest rate. Some borrowers later stopped paying and the loan was written off (**charged off**). The lender loses money on each of those.

**Question:** using only what the lender knows *when the application is decided*, can we estimate how likely the loan is to be charged off?

### 1.2 Input and output

| | Description |
|---|---|
| **Input** | One row per loan application: amount, term, interest rate and grade assigned by the lender, employment length, home ownership, income, purpose, debt-to-income ratio, and credit-report summaries (FICO score, open accounts, credit enquiries, ...) |
| **Output** | A **score** between 0 and 1: the estimated probability that the loan is charged off. A threshold turns the score into a decision: *flag as high risk* or *pass*. |
| **Label (for learning)** | `1` if the loan was `Charged Off`, `0` if `Fully Paid` |

### 1.3 Why it matters, and what each mistake costs

| Mistake | What happens | Cost to the lender |
|---|---|---|
| **Missed default** (false negative) | A loan that will default is approved | Lose part of the principal |
| **False alarm** (false positive) | A loan that would have been repaid is rejected or priced higher | Lose the interest it would have earned |

The dataset records no dollar figure for either mistake, so **we do not invent one**. Later we show the trade-off between the two mistakes and state that the threshold choice is a business decision.

### 1.4 What already exists? (the incumbent)
Lending Club already assigns every loan a **grade** (A-G) and **sub-grade** (A1-G5) when it decides. These are an existing risk ranking. A new model is only useful if it ranks risk **better than the sub-grade does**. So the headline of this project is the *improvement over the sub-grade*, not the raw accuracy of the model.

## Step 2 - Define the problem type, target and metrics

### 2.1 Problem type
**Binary classification** (two outcomes: default or not), where we want a *probability* for each loan.

### 2.2 Target rule
- `default = 1` when `loan_status == "Charged Off"`
- `default = 0` when `loan_status == "Fully Paid"`
- Loans that are still running (`Current`, `Late`, ...) are **dropped**, because their final outcome is not known yet. (Notebook 02 counts how many loans this removes and checks whether it distorts the labels.)

### 2.3 Baselines

| # | Baseline | Meaning |
|---|---|---|
| **B0** | Majority class | Predict "no default" for everyone |
| **B1** | Lending Club sub-grade | Use the existing sub-grade as the risk score |

### 2.4 Metrics (and why not plain accuracy)
The full definitions, with when each is appropriate, are in Table 1 of notebook 03. The most important point is shown with a small invented example below: **accuracy can look fine while the model is useless.**

**Table 1.** Toy example: 100 loans, 20 of which default. Two "models" with the same accuracy.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score

# 100 invented loans: first 20 default (1), last 80 are repaid (0)
actual = np.array([1] * 20 + [0] * 80)

# Model L ("lazy"): says nobody defaults
lazy = np.zeros(100, dtype=int)

# Model U ("useful"): flags 30 loans. It catches 15 of the 20 defaulters and wrongly flags 15 good loans.
useful = np.array([1] * 15 + [0] * 5 + [1] * 15 + [0] * 65)

toy = pd.DataFrame({name: {"Accuracy": accuracy_score(actual, pred),
                           "Precision": precision_score(actual, pred, zero_division=0),
                           "Recall": recall_score(actual, pred)}
                    for name, pred in {"Lazy (nobody defaults)": lazy, "Useful (flags 30 loans)": useful}.items()}).T
toy

,Accuracy,Precision,Recall
Lazy (nobody defaults),0.8,0.0,0.00
Useful (flags 30 loans),0.8,0.5,0.75


In [2]:
print(f"Both models have accuracy {toy['Accuracy'].iloc[0]:.2f} and {toy['Accuracy'].iloc[1]:.2f}.")
print(f"But the lazy model catches {toy['Recall'].iloc[0]:.0%} of defaulters and the useful one catches {toy['Recall'].iloc[1]:.0%}.")
print("Action -> we do not use accuracy as the headline metric. We use ROC-AUC and PR-AUC (threshold-free), "
      "and precision / recall / F1 at a chosen threshold.")

Both models have accuracy 0.80 and 0.80.
But the lazy model catches 0% of defaulters and the useful one catches 75%.
Action -> we do not use accuracy as the headline metric. We use ROC-AUC and PR-AUC (threshold-free), and precision / recall / F1 at a chosen threshold.


### 2.5 How we will split and validate

| Decision | Choice | Reason |
|---|---|---|
| Train / test split | **Out of time**: loans issued before 2016-01-01 are train, on or after are test | A lender scores *future* applicants with a model built on *past* loans. The cutoff is fixed and shared by all groups using this dataset. Notebook 03 (section 9.7) measures what a random split would have hidden. |
| Validation | `TimeSeriesSplit` on training rows ordered by issue date | Shuffled folds would let the model train on later loans and test on earlier ones |
| Preprocessing | Inside a scikit-learn `Pipeline`, fitted on training rows only | Medians, means and category lists are learned from data; learning them from test rows would leak test information |
| Features | Only columns known at application time | Columns recorded after the loan starts reveal the answer. Notebook 02 tests this idea with code. |
| Test set | Opened **once**, after all choices are fixed | Tuning on the test set would make its result meaningless |

### 2.6 Project map

| Notebook | Steps | Produces |
|---|---|---|
| 01 Problem framing (this one) | 1-2 | Decisions |
| 02 Data and features | 3-7 | `clean.parquet`, `split_manifest.json`, Figures 1-5 |
| 03 Modeling and evaluation | 8-10 | `model.joblib`, `metrics.csv`, Figures 6-13 |
| 04 Packaging and monitoring | 11 | `model_card.json`, Figures 14-15 |

**Run order:** 01 -> 02 -> 03 -> 04, each from top to bottom.